# 09 — EXP_16: Final Synthesis (Phase 9)

**Goal.** Aggregate every architecture's measured metrics into a single weighted ranking row, per [plan.md §11](../plan.md#11-phase-9--exp_16-final-synthesis):

$$\text{final\_score} = 0.25 \cdot \text{Accuracy} + 0.25 \cdot \text{Faithfulness} + 0.20 \cdot \text{Retrieval} + 0.15 \cdot \text{Safety} + 0.10 \cdot \text{Explainability} + 0.05 \cdot \text{Latency}$$

All component scores normalised to [0, 1]. Lower-is-better metrics (latency) are inverted via in-set min-max scaling.

**Cost:** $0 (pure aggregation; no LLM calls).

**Tables filled:** Table 10 (Adaptive vs Best Fixed) · Table 12 (Final Weighted Ranking).

**Inputs (all on disk from Phases 4–8):**
- `results/exp_0[1-5]__test_1273/summary.json` — Accuracy + mean_latency
- `results/exp_0[1-5]__golden_234/{summary.json, ragas_scores.csv}` — RAGAS metrics
- `results/exp_07_adaptive_variant_{a,b}__test_1273/summary.json` — Adaptive accuracy + latency
- `results/exp_12_agreement/stage_b_retrievalchanged_mhop.jsonl` — Multi-Hop LIME-SHAP agreement (Adaptive variants inherit by routing share; other fixed archs have no measurement)
- `data/processed/complexity_labels.parquet` — for Adaptive RAGAS score-join

**Two judgement calls baked into the normaliser** (documented in [`src/synthesis/normaliser.py`](../src/synthesis/normaliser.py)):
1. **NoRAG safety = 0** (no chunks → RAGAS Hallucination_Rate is structurally undefined; conservative zero-floor is honest about the measurement gap).
2. **Explainability for non-Multi-Hop fixed archs = 0** (LIME/SHAP was only measured on the retrieval-changed Multi-Hop surface in Phase 6; Adaptive variants inherit Multi-Hop's signal weighted by routing share).

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.synthesis import (
    DEFAULT_WEIGHTS,
    collect_architecture_metrics,
    normalise,
    pareto_frontier,
    use_case_recommendations,
    weighted_score,
)

OUT_DIR = REPO_ROOT / "results" / "exp_16_final_synthesis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUT_DIR)
print("Default weights :", DEFAULT_WEIGHTS, "(sum=", sum(DEFAULT_WEIGHTS.values()), ")")

Output directory: /Users/rajak/Workstation/Projects/myGitHub/thesis-project/results/exp_16_final_synthesis
Default weights : {'Accuracy': 0.25, 'Faithfulness': 0.25, 'Retrieval': 0.2, 'Safety': 0.15, 'Explainability': 0.1, 'Latency': 0.05} (sum= 1.0 )


## 2. Stage A — Collect raw per-architecture metrics

Reads every `summary.json` and the EXP_12 agreement file. Adaptive RAGAS is re-derived via the per-question score-join (the logic from [`notebooks/05_exp07_adaptive_rag.ipynb` §8](05_exp07_adaptive_rag.ipynb), now in [`src/synthesis/aggregator.py`](../src/synthesis/aggregator.py)).

In [2]:
raw = collect_architecture_metrics(REPO_ROOT)
raw.to_csv(OUT_DIR / "component_scores_raw.csv", index=False)
raw.round(4)

,architecture,accuracy_test_1273,mean_latency_s_test_1273,groq_calls_per_q,faithfulness_golden_234,hallucination_rate_golden_234,context_precision_golden_234,context_recall_golden_234,answer_correctness_golden_234,answer_relevance_golden_234,lime_shap_spearman,n_explainability_questions
0,NoRAG,0.7738,0.2956,1.000,NaN,NaN,NaN,NaN,0.8738,0.5977,NaN,0
1,Naive,0.7573,0.4437,1.000,0.1308,0.8957,0.3285,0.4124,0.8376,0.5961,NaN,0
2,Sparse,0.7581,0.4354,1.000,0.0401,0.9657,0.0811,0.1073,0.8384,0.5971,NaN,0
3,Hybrid,0.7659,0.4432,1.000,0.0944,0.9174,0.2797,0.3483,0.8273,0.5966,NaN,0
4,MultiHop,0.7958,0.6600,3.000,0.2833,0.7371,0.3737,0.7115,0.8685,0.5955,0.6325,134
5,Adaptive_A,0.7863,0.6964,1.806,0.1966,0.8369,0.3598,0.5705,0.8473,0.5971,0.2549,134
6,Adaptive_B,0.7832,0.5740,2.425,0.2756,0.7529,0.3792,0.7544,0.8669,0.5932,0.4506,134


**Cross-checks against the prior output notes:**
- NoRAG accuracy on test_1273 = **0.7738** (matches `04a_exp01_output.md`).
- Multi-Hop accuracy on test_1273 = **0.7958** (matches `04e_exp05_output.md`).
- Multi-Hop Faithfulness on golden_234 = **0.2833** (matches `04e_exp05_output.md`).
- Variant A score-joined Faithfulness = **0.197** (matches `05_exp07_output.md` §3.4).
- Variant B score-joined Faithfulness = **0.276** (matches `05_exp07_output.md` §3.4).
- Multi-Hop LIME-SHAP Spearman ρ mean = **~0.632** (matches `06_exp10_11_12_output.md` §2.7).

## 3. Stage B — Normalise to [0, 1] component scores

In [3]:
norm = normalise(raw)
norm.to_csv(OUT_DIR / "component_scores_normalised.csv", index=False)
norm.round(4)

,architecture,Accuracy,Faithfulness,Retrieval,Safety,Explainability,Latency
0,NoRAG,0.7738,0.0000,0.0000,0.0000,0.0000,1.0000
1,Naive,0.7573,0.1308,0.3705,0.1043,0.0000,0.6306
2,Sparse,0.7581,0.0401,0.0942,0.0343,0.0000,0.6512
3,Hybrid,0.7659,0.0944,0.3140,0.0826,0.0000,0.6316
4,MultiHop,0.7958,0.2833,0.5426,0.2629,0.6325,0.0909
5,Adaptive_A,0.7863,0.1966,0.4652,0.1631,0.2549,0.0000
6,Adaptive_B,0.7832,0.2756,0.5668,0.2471,0.4506,0.3054


## 4. Stage C — Apply weights and rank

The plan §11 weights are the canonical setting; the `final_score` column is the weighted sum, `rank` is descending order (1 = best balanced).

In [4]:
ranked = weighted_score(norm)
ranked.to_csv(OUT_DIR / "table12_final_ranking.csv", index=False)

print("=== Table 12 — Final Weighted Ranking (plan §11 weights) ===")
print(ranked.round(4).to_string(index=False))

=== Table 12 — Final Weighted Ranking (plan §11 weights) ===
architecture  Accuracy  Faithfulness  Retrieval  Safety  Explainability  Latency  final_score  rank
    MultiHop    0.7958        0.2833     0.5426  0.2629          0.6325   0.0909       0.4855     1
  Adaptive_B    0.7832        0.2756     0.5668  0.2471          0.4506   0.3054       0.4755     2
  Adaptive_A    0.7863        0.1966     0.4652  0.1631          0.2549   0.0000       0.3887     3
       Naive    0.7573        0.1308     0.3705  0.1043          0.0000   0.6306       0.3433     4
      Hybrid    0.7659        0.0944     0.3140  0.0826          0.0000   0.6316       0.3218     5
      Sparse    0.7581        0.0401     0.0942  0.0343          0.0000   0.6512       0.2561     6
       NoRAG    0.7738        0.0000     0.0000  0.0000          0.0000   1.0000       0.2434     7


## 5. Stage D — Pareto frontier on (accuracy, Groq calls/Q)

The weighted ranking collapses six dimensions into one score; the Pareto frontier on the two most-cited operational axes (accuracy and compute cost) is a sanity check that the ranking respects multi-objective trade-offs.

In [5]:
pareto = pareto_frontier(raw)
pareto[["architecture", "accuracy_test_1273", "groq_calls_per_q", "pareto_status"]].to_csv(
    OUT_DIR / "pareto_status.csv", index=False
)
print("=== Pareto frontier — accuracy vs Groq calls/Q ===")
print(
    pareto[["architecture", "accuracy_test_1273", "groq_calls_per_q", "pareto_status"]]
    .round(4)
    .to_string(index=False)
)

=== Pareto frontier — accuracy vs Groq calls/Q ===
architecture  accuracy_test_1273  groq_calls_per_q pareto_status
       NoRAG              0.7738             1.000      frontier
       Naive              0.7573             1.000     DOMINATED
      Sparse              0.7581             1.000     DOMINATED
      Hybrid              0.7659             1.000     DOMINATED
    MultiHop              0.7958             3.000      frontier
  Adaptive_A              0.7863             1.806      frontier
  Adaptive_B              0.7832             2.425     DOMINATED


## 6. Stage E — Use-case recommendations (plan §11)

Pick one architecture per deployment scenario from the data.

In [6]:
recs = use_case_recommendations(raw, ranked)
recs.to_csv(OUT_DIR / "recommendations.csv", index=False)
print("=== Use-case recommendations ===")
print(recs.to_string(index=False))

=== Use-case recommendations ===
              use_case architecture                        metric    value                                                                                                                                       rationale
           Lowest cost        NoRAG              groq_calls_per_q 1.000000                                                                                           Fewest Groq calls per question (cheapest deployment).
      Highest accuracy     MultiHop            accuracy_test_1273 0.795758                                                                           Top exact-match on the 1,273-question contamination-clean test split.
  Lowest hallucination     MultiHop hallucination_rate_golden_234 0.737069                                                               Lowest RAGAS Hallucination_Rate (fraction with Faithfulness < 0.5) on golden_234.
Highest explainability     MultiHop            lime_shap_spearman 0.632469 Highest LIME-SHA

## 7. Stage F — Table 10 (Adaptive vs Best Fixed)

Side-by-side comparison of the two Adaptive variants against the best fixed architecture (Multi-Hop). Δ columns: positive means the adaptive variant beats Multi-Hop on that dimension.

In [7]:
raw_by_arch = raw.set_index("architecture")
norm_by_arch = norm.set_index("architecture")

rows = []
metric_specs = [
    ("Accuracy (test_1273)", "accuracy_test_1273", raw_by_arch, "higher"),
    ("Faithfulness (golden_234)", "faithfulness_golden_234", raw_by_arch, "higher"),
    ("Context Precision (golden_234)", "context_precision_golden_234", raw_by_arch, "higher"),
    ("Context Recall (golden_234)", "context_recall_golden_234", raw_by_arch, "higher"),
    ("Hallucination Rate (golden_234)", "hallucination_rate_golden_234", raw_by_arch, "lower"),
    ("LIME-SHAP Spearman ρ", "lime_shap_spearman", raw_by_arch, "higher"),
    ("Mean latency (s, test_1273)", "mean_latency_s_test_1273", raw_by_arch, "lower"),
    ("Groq calls / Q (test_1273)", "groq_calls_per_q", raw_by_arch, "lower"),
    ("final_score (weighted)", "final_score", ranked.set_index("architecture"), "higher"),
]

for label, col, source, direction in metric_specs:
    mh = float(source.loc["MultiHop", col])
    a = float(source.loc["Adaptive_A", col])
    b = float(source.loc["Adaptive_B", col])
    sign = 1.0 if direction == "higher" else -1.0
    rows.append(
        {
            "metric": label,
            "MultiHop": round(mh, 4),
            "Adaptive_A": round(a, 4),
            "Adaptive_B": round(b, 4),
            "Δ_A_vs_MH": round(sign * (a - mh), 4),
            "Δ_B_vs_MH": round(sign * (b - mh), 4),
            "direction": direction,
        }
    )

table10 = pd.DataFrame(rows)
table10.to_csv(OUT_DIR / "table10_adaptive_vs_fixed.csv", index=False)
print("=== Table 10 — Adaptive vs Best Fixed (Multi-Hop) ===")
print(table10.to_string(index=False))

=== Table 10 — Adaptive vs Best Fixed (Multi-Hop) ===
                         metric  MultiHop  Adaptive_A  Adaptive_B  Δ_A_vs_MH  Δ_B_vs_MH direction
           Accuracy (test_1273)    0.7958      0.7863      0.7832    -0.0094    -0.0126    higher
      Faithfulness (golden_234)    0.2833      0.1966      0.2756    -0.0868    -0.0077    higher
 Context Precision (golden_234)    0.3737      0.3598      0.3792    -0.0139     0.0055    higher
    Context Recall (golden_234)    0.7115      0.5705      0.7544    -0.1410     0.0428    higher
Hallucination Rate (golden_234)    0.7371      0.8369      0.7529    -0.0998    -0.0159     lower
           LIME-SHAP Spearman ρ    0.6325      0.2549      0.4506    -0.3776    -0.1818    higher
    Mean latency (s, test_1273)    0.6600      0.6964      0.5740    -0.0364     0.0860     lower
     Groq calls / Q (test_1273)    3.0000      1.8060      2.4250     1.1940     0.5750     lower
         final_score (weighted)    0.4855      0.3887      0.475

## 8. Stage G — Sensitivity analysis: rank stability under weight perturbations

The plan §11 weights are a fixed default. A reviewer will ask: *if you re-weighted, would the headline change?* Re-run the ranking under three alternative weight regimes:
- **Accuracy-heavy** (0.50 acc · everything else proportional)
- **Safety-heavy** (0.30 safety + 0.30 faithfulness · everything else proportional)
- **Compute-heavy** (0.20 latency · everything else proportional)

Rank-1 stability under these perturbations tells us how load-bearing the weight choice is.

In [8]:
def renormalise(weights):
    total = sum(weights.values())
    return {k: v / total for k, v in weights.items()}

regimes = {
    "plan_default": DEFAULT_WEIGHTS,
    "accuracy_heavy": renormalise({"Accuracy": 0.50, "Faithfulness": 0.15, "Retrieval": 0.12, "Safety": 0.10, "Explainability": 0.08, "Latency": 0.05}),
    "safety_heavy": renormalise({"Accuracy": 0.15, "Faithfulness": 0.30, "Retrieval": 0.15, "Safety": 0.30, "Explainability": 0.05, "Latency": 0.05}),
    "compute_heavy": renormalise({"Accuracy": 0.20, "Faithfulness": 0.20, "Retrieval": 0.15, "Safety": 0.15, "Explainability": 0.10, "Latency": 0.20}),
}

sensitivity_rows = []
for regime_name, weights in regimes.items():
    r = weighted_score(norm, weights=weights)
    for _, row in r.iterrows():
        sensitivity_rows.append(
            {
                "regime": regime_name,
                "architecture": row["architecture"],
                "final_score": round(float(row["final_score"]), 4),
                "rank": int(row["rank"]),
            }
        )

sens = pd.DataFrame(sensitivity_rows)
sens_wide = sens.pivot(index="architecture", columns="regime", values="rank").reindex(
    ["MultiHop", "Adaptive_B", "Adaptive_A", "Naive", "Hybrid", "Sparse", "NoRAG"]
)
sens_wide.to_csv(OUT_DIR / "sensitivity_ranks.csv")
print("=== Rank stability across weight regimes ===")
print(sens_wide.to_string())

=== Rank stability across weight regimes ===
regime        accuracy_heavy  compute_heavy  plan_default  safety_heavy
architecture                                                           
MultiHop                   1              2             1             1
Adaptive_B                 2              1             2             2
Adaptive_A                 3              6             3             3
Naive                      4              3             4             4
Hybrid                     5              4             5             5
Sparse                     7              7             6             6
NoRAG                      6              5             7             7


## 9. Stage H — Write the Phase-9 summary.json (paste-into-Excel anchor)

In [9]:
from datetime import datetime, timezone

top = ranked.iloc[0]
summary = {
    "experiment_id": "EXP_16_FINAL_SYNTHESIS",
    "phase": 9,
    "dataset": "aggregated",
    "n_architectures": int(len(ranked)),
    "weights": DEFAULT_WEIGHTS,
    "weight_sum_check": round(sum(DEFAULT_WEIGHTS.values()), 6),
    "top_ranked": {
        "architecture": str(top["architecture"]),
        "final_score": round(float(top["final_score"]), 4),
    },
    "ranking": [
        {"rank": int(r["rank"]), "architecture": str(r["architecture"]), "final_score": round(float(r["final_score"]), 4)}
        for _, r in ranked.iterrows()
    ],
    "recommendations": {
        str(r["use_case"]): {
            "architecture": str(r["architecture"]),
            "metric": str(r["metric"]),
            "value": round(float(r["value"]), 4) if pd.notna(r["value"]) else None,
        }
        for _, r in recs.iterrows()
    },
    "pareto_frontier": pareto[pareto.pareto_status == "frontier"]["architecture"].tolist(),
    "sensitivity_top_rank_by_regime": {
        regime: sens_wide[regime].idxmin() for regime in sens_wide.columns
    },
    "normalisation_scheme": "raw_with_minmax_latency (RAGAS-style metrics native [0,1]; latency min-max-inverted in-set; NaN floors to 0)",
    "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}

(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "experiment_id": "EXP_16_FINAL_SYNTHESIS",
  "phase": 9,
  "dataset": "aggregated",
  "n_architectures": 7,
  "weights": {
    "Accuracy": 0.25,
    "Faithfulness": 0.25,
    "Retrieval": 0.2,
    "Safety": 0.15,
    "Explainability": 0.1,
    "Latency": 0.05
  },
  "weight_sum_check": 1.0,
  "top_ranked": {
    "architecture": "MultiHop",
    "final_score": 0.4855
  },
  "ranking": [
    {
      "rank": 1,
      "architecture": "MultiHop",
      "final_score": 0.4855
    },
    {
      "rank": 2,
      "architecture": "Adaptive_B",
      "final_score": 0.4755
    },
    {
      "rank": 3,
      "architecture": "Adaptive_A",
      "final_score": 0.3887
    },
    {
      "rank": 4,
      "architecture": "Naive",
      "final_score": 0.3433
    },
    {
      "rank": 5,
      "architecture": "Hybrid",
      "final_score": 0.3218
    },
    {
      "rank": 6,
      "architecture": "Sparse",
      "final_score": 0.2561
    },
    {
      "rank": 7,
      "architecture": "NoRAG",
     

## 10. Headline take-aways

1. **Multi-Hop ranks #1** under the plan §11 default weights (final_score ≈ 0.49). The runner-up is **Adaptive Variant B** at ≈ 0.48 — about 1 score-point behind, with strictly higher Latency component (Variant B is 13 % faster on mean latency, but Multi-Hop dominates on accuracy + explainability).

2. **The proposal's expected winner (Adaptive Variant A) places #3.** Adaptive Variant A is the cost-efficient Pareto-frontier point on accuracy-vs-Groq-calls (see notebook 05 §9), but on the six-dimension weighted ranking it is hurt by Variant A routing Moderate → Hybrid (Faithfulness 0.09) instead of Variant B's Moderate → Multi-Hop (Faithfulness 0.28). Plan §11's 0.25 weight on Faithfulness is the main reason Variant B beats Variant A in the synthesis.

3. **NoRAG places last (rank 7)** despite having the second-highest test accuracy. The structural zero on Faithfulness, Retrieval, Safety, and Explainability (un-measurable on a no-chunks architecture) is the right honest penalty — the thesis claim is that grounded-correct is materially better than memorised-correct, and the weighting reflects that.

4. **Use-case picks** (from `recommendations.csv`): cheapest = NoRAG · highest acc = Multi-Hop · lowest hallucination = Multi-Hop · highest explainability = Multi-Hop · best balanced = Multi-Hop. Multi-Hop wins four of five use cases — the proposal's framing of *"Adaptive should be the best balanced"* is **not** supported on the locked plan §11 weights. A re-weighted ranking that prioritises compute (compute-heavy regime in §8) shifts the headline; document the sensitivity honestly.

5. **Files written to `results/exp_16_final_synthesis/`:**
   - `table12_final_ranking.csv` — Excel Table 12 paste-target
   - `table10_adaptive_vs_fixed.csv` — Excel Table 10 paste-target
   - `component_scores_raw.csv` — full intermediate (per-arch raw metrics)
   - `component_scores_normalised.csv` — full intermediate (per-arch [0,1] components)
   - `pareto_status.csv` — frontier sanity check
   - `recommendations.csv` — use-case mapping
   - `sensitivity_ranks.csv` — rank stability under 3 alternative weight regimes
   - `summary.json` — paste-into-Excel anchor + the publishable headlines